In [ ]:
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 68.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 65.4 MB/s eta 0:00:00


In [ ]:
!pip install -U git+https://github.com/luca-medeiros/lang-segment-anything.git

  Cloning https://github.com/luca-medeiros/lang-segment-anything.git to /tmp/pip-req-build-n0sstru6
  Running command git clone --filter=blob:none --quiet https://github.com/luca-medeiros/lang-segment-anything.git /tmp/pip-req-build-n0sstru6
  Resolved https://github.com/luca-medeiros/lang-segment-anything.git to commit 918043ed4666eea04da88aa179eb8d27ef4b1a1d
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Cloning https://github.com/facebookresearch/segment-anything-2 (to revision c2ec8e14a185632b0a5d8b161928ceb50197eddc) to /tmp/pip-install-guliezs5/sam-2_92d6c8f0ae444fa3a521e144158dc23e
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/segment-anything-2 /tmp/pip-install-guliezs5/sam-2_92d6c8f0ae444fa3a521e144158dc23e
  Running command git rev-parse -q --verify 'sha^c2ec8e14a185632b0a5d8b161928ceb50197eddc'
  Running command git fetch -q https://github.co

In [41]:
%%writefile app.py
import streamlit as st
import torch
import numpy as np
from PIL import Image
from lang_sam import LangSAM
from diffusers import StableDiffusionInpaintPipeline
import os
os.environ['HF_HUB_OFFLINE'] = '0'
os.environ['TRANSFORMERS_OFFLINE'] = '0'

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

@st.cache_resource
def load_models():
    sam_model = LangSAM()
    pipe = StableDiffusionInpaintPipeline.from_pretrained(
        "sd-legacy/stable-diffusion-inpainting",
        torch_dtype=torch.float16)
    pipe.to(device)
    return sam_model, pipe

# UI Setup
st.title("Prompt based Inpainting Pipeline")
uploaded_file = st.file_uploader("Image", type=["jpg", "png", "jpeg"])
src_text = st.text_input("Object", "sunglasses")
tgt_text = st.text_input("Replace with", "goggles")

if uploaded_file:
    img = Image.open(uploaded_file).convert("RGB").resize((512, 512))

    col1, col2 = st.columns(2)
    with col1:
        st.subheader("Input Image")
        st.image(img, use_container_width=True)

    if st.button("Run Logic"):
        sam, pipe = load_models()

        # Step 1: LangSAM predict
        results = sam.predict([img], [src_text])
        result = results[0]

        masks = result["masks"]
        boxes = result["boxes"]
        labels = result["labels"]
        logits = result["scores"]
        mask_scores = result['mask_scores']

        # Step 2: Select best mask based on LangSAM masking
        best_idx = np.argmax(mask_scores)
        mask = masks[best_idx]

        # Step 3: Convert mask to PIL Image
        mask = (mask > 0.5).astype(np.uint8) * 255
        mask_pil = Image.fromarray(mask)

        # Step 4: Perform inpainting with StableDiffusionInpaintPipeline
        inpainted_image = pipe(prompt=tgt_text, image=[img], mask_image=mask_pil).images[0]

        with col2:
                st.subheader("Result")
                st.image(inpainted_image, use_container_width=True)


Overwriting app.py


In [42]:
from google.colab.output import eval_js

# Get your proxy URL
print("Click here to open the app:")
print(eval_js("google.colab.kernel.proxyPort(8501)"))

# Run without forcing offline mode so it can resolve the "snapshot" error
!streamlit run app.py --server.port 8501 --server.headless True --server.enableCORS False --server.enableXsrfProtection False

Click here to open the app:
https://8501-gpu-t4-s-kkb-usw1b0-2vvhw8nto8xj1-b.us-west1-0.prod.colab.dev


2026-05-04 18:51:58.727 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://136.118.174.179:8501

/usr/local/lib/python3.12/dist-packages/sam2/modeling/sam/transformer.py:23: UserWarning: Flash Attention is disabled as it requires a GPU with Ampere (8.0) CUDA capability.
  OLD_GPU, USE_FLASH_ATTN, MATH_KERNEL_ON = get_sdpa_settings()
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
2026-05-04 18:52:30.833 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 